# Customer Churn Prediction - Model Tuning

##  Hyperparameter Tuning

In this notebook, hyperparameter tuning is performed to improve model
performance and select a final model for deployment.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import joblib

In [2]:
df = pd.read_csv("../data/customer_churn_cleaned.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = df.drop("Churn", axis=1)
y = df["Churn"]

In [4]:
categorical_cols = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

C:\Users\A C E R\AppData\Local\Temp\ipykernel_21528\4285972795.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [5]:
X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

In [8]:
log_params = {
    "C": [0.01, 0.1, 1, 10],
    "class_weight": [None, "balanced"]
}

In [9]:
log_grid = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    param_grid=log_params,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced']}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [10]:
print("Best parameters:")
print(log_grid.best_params_)

print("Best CV ROC-AUC:")
print(log_grid.best_score_)

Best parameters:
{'C': 10, 'class_weight': None}
Best CV ROC-AUC:
0.8459388437649308


In [11]:
tuned_log = log_grid.best_estimator_

tuned_log_pred = tuned_log.predict(X_test)
tuned_log_prob = tuned_log.predict_proba(X_test)[:, 1]

In [12]:
print(
    "Accuracy:",
    round(accuracy_score(y_test, tuned_log_pred), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, tuned_log_pred), 4)
)

print(
    "Recall:",
    round(recall_score(y_test, tuned_log_pred), 4)
)

print(
    "F1:",
    round(f1_score(y_test, tuned_log_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, tuned_log_prob), 4)
)

Accuracy: 0.801
Precision: 0.6407
Recall: 0.5722
F1: 0.6045
ROC-AUC: 0.8353


In [13]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "class_weight": [None, "balanced"]
}

In [14]:
rf_grid = GridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_params,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': [None, 'balanced'], 'max_depth': [5, 10, ...], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [15]:
print("Best parameters:")
print(rf_grid.best_params_)

print("Best CV ROC-AUC:")
print(rf_grid.best_score_)

Best parameters:
{'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}
Best CV ROC-AUC:
0.844286038206451


In [16]:
tuned_rf = rf_grid.best_estimator_

tuned_rf_pred = tuned_rf.predict(X_test)
tuned_rf_prob = tuned_rf.predict_proba(X_test)[:, 1]

In [17]:
print(
    "Accuracy:",
    round(accuracy_score(y_test, tuned_rf_pred), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, tuned_rf_pred), 4)
)

print(
    "Recall:",
    round(recall_score(y_test, tuned_rf_pred), 4)
)

print(
    "F1:",
    round(f1_score(y_test, tuned_rf_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, tuned_rf_prob), 4)
)

Accuracy: 0.7242
Precision: 0.4886
Recall: 0.8048
F1: 0.6081
ROC-AUC: 0.8356


In [18]:
tuned_results = pd.DataFrame({
    "Model": [
        "Tuned Logistic Regression",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, tuned_log_pred),
        accuracy_score(y_test, tuned_rf_pred)
    ],
    "Precision": [
        precision_score(y_test, tuned_log_pred),
        precision_score(y_test, tuned_rf_pred)
    ],
    "Recall": [
        recall_score(y_test, tuned_log_pred),
        recall_score(y_test, tuned_rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, tuned_log_pred),
        f1_score(y_test, tuned_rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, tuned_log_prob),
        roc_auc_score(y_test, tuned_rf_prob)
    ]
})

tuned_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Tuned Logistic Regression,0.800995,0.640719,0.572193,0.604520,0.835336
1,Tuned Random Forest,0.724236,0.488636,0.804813,0.608081,0.835639


In [19]:
final_model = tuned_rf

### Final Model Selection

The Tuned Random Forest was selected for the deployment stage because the
project prioritizes identifying customers who are at risk of churn.

The model achieved a recall of **80.48%**, meaning it identified a large
proportion of the customers who actually churned. Its ROC-AUC was **0.8356**
and its F1-score was **60.81%**.

This choice involves a trade-off: the model's precision was **48.86%**, so
it produces more false-positive churn alerts. For a retention use case,
this may be acceptable when missing an actual churn customer is considered
more costly than reviewing an additional at-risk customer.

In [20]:
joblib.dump(
    final_model,
    "../model/churn_model.pkl"
)

joblib.dump(
    scaler,
    "../model/scaler.pkl"
)

joblib.dump(
    X.columns.tolist(),
    "../model/model_columns.pkl"
)

joblib.dump(
    numerical_cols,
    "../model/numerical_columns.pkl"
)

['../model/numerical_columns.pkl']

In [21]:
loaded_model = joblib.load(
    "../model/churn_model.pkl"
)

loaded_model

,n_estimators,200
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


##  Model Tuning Results

Hyperparameter tuning was performed using 5-fold cross-validation.

### Tuned Logistic Regression

- Best Parameters: `...`
- Accuracy: ...
- Precision: ...
- Recall: ...
- F1-score: ...
- ROC-AUC: ...

### Tuned Random Forest

- Best Parameters: `...`
- Accuracy: ...
- Precision: ...
- Recall: ...
- F1-score: ...
- ROC-AUC: ...

### Deployment Model

The deployment model was selected based on the trade-offs between ROC-AUC,
F1-score, recall, precision, and the objective of identifying customers at
risk of churn.

The trained model and preprocessing artifacts were saved for use in the
prediction application.